In [ ]:
#0 Load Libraries and Configurations

import os
import wrds
import pandas as pd
import numpy as np

# ---------- User-configurable paths ----------
PATH_DATA_INTERMEDIATE = "/Users/nglei/Desktop/Academics/SMU/Modules/QF600 Asset Pricing/Project/Project Code/cz_data/intermediate"  # <-- change this
os.makedirs(PATH_DATA_INTERMEDIATE, exist_ok=True)

OUT_PARQUET = os.path.join(PATH_DATA_INTERMEDIATE, "IBES_EPS_Unadj.parquet")
OUT_CSV     = os.path.join(PATH_DATA_INTERMEDIATE, "IBES_EPS_Unadj.csv")

In [ ]:
#1 Load IBES Data

SQL = """
SELECT
    a.ticker,
    a.statpers,
    a.measure,
    a.fpi,
    a.numest,
    a.medest,
    a.meanest,
    a.stdev,
    a.fpedats
FROM ibes.statsumu_epsus AS a
WHERE a.fpi IN ('0','1','2','6')
  AND a.statpers >= DATE '2000-01-01'
;
"""

In [ ]:
#2 IBES Data Extraction From WRDS

db = wrds.Connection()
df = db.raw_sql(SQL, date_cols=["statpers", "fpedats"])

In [ ]:
#Data Cleaning

# ---------------- Set up linking variables ----------------
# time_avail_m = month(statpers)
df["time_avail_m"] = df["statpers"].dt.to_period("M").dt.to_timestamp("MS")

# rename ticker
df = df.rename(columns={"ticker": "tickerIBES"})

# drop measure
if "measure" in df.columns:
    df = df.drop(columns=["measure"])

# ---------------- Keep last obs each month ----------------
# Drop rows with missing meanest
df = df[df["meanest"].notna()].copy()

# Sort, then keep the last statpers within (tickerIBES, fpi, time_avail_m)
df = df.sort_values(["tickerIBES", "fpi", "time_avail_m", "statpers"], kind="mergesort")
df = df.groupby(["tickerIBES", "fpi", "time_avail_m"], as_index=False).tail(1)

# Optional: tidy dtypes
for col in ["numest", "medest", "meanest", "stdev"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# ---------------- Save ----------------
df.to_parquet(OUT_PARQUET, index=False)
df.to_csv(OUT_CSV, index=False)

print("Saved:")
print(" -", OUT_PARQUET)
print(" -", OUT_CSV)
print(df.head())